# DabReels: free AI video test (Wan, open source)

Generates the 6-shot test scene of **وريث الرياض** with free, open-source models (Apache 2.0, commercial use allowed).

**Before running:** Settings (right panel) → Accelerator: **GPU T4 x2** · Internet: **On**. Then **Run All**.
Results appear in the Output tab as `dabreels_test.zip` (clips + timing report).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U "git+https://github.com/huggingface/diffusers" transformers accelerate ftfy imageio imageio-ffmpeg sentencepiece

In [ ]:
SCENE = {
 "series": "warith_riad",
 "title_ar": "وريث الرياض",
 "test": "Pilot test scene, episode 1 opening (about 25 seconds)",
 "style": "cinematic vertical 9:16 short drama, warm golden lantern light, shallow depth of field, realistic, film grain",
 "characters": {
  "youssef": "Youssef, a 25-year-old Moroccan man, short dark curly hair, light trimmed beard, warm brown eyes, crisp white shirt with a burgundy waiter vest and a small brass name tag",
  "guest": "a wealthy man in his fifties, slicked grey hair, expensive grey suit, gold watch, arrogant smile",
  "lalla_zahra": "Lalla Zahra, a Moroccan housekeeper in her seventies, white headscarf, kind wrinkled face, dark green kaftan"
 },
 "setting": "a luxury riad courtyard in the Marrakech medina at dusk, zellige tiles, a marble fountain, orange trees, glowing brass lanterns, carved cedar arches",
 "negative": "blurry, low quality, distorted face, extra fingers, deformed hands, text, watermark, logo, subtitles, cartoon, oversaturated, flicker, jpeg artifacts",
 "shots": [
  {
   "id": 1,
   "prompt": "Slow aerial push-in over {setting}. Lanterns flicker, water sparkles in the fountain, guests at tables in the distance.",
   "narration_ar": "في قلب مدينة مراكش، يقف رياضٌ فاخر لا يدخله إلا الأغنياء."
  },
  {
   "id": 2,
   "prompt": "{youssef} walks through {setting} carrying a silver tray of mint tea glasses, polite nervous smile, camera tracking beside him.",
   "narration_ar": "يوسف، نادلٌ بسيط، يعمل هنا منذ ثلاث سنوات."
  },
  {
   "id": 3,
   "prompt": "At a candle-lit table in {setting}, {guest} laughs mockingly and points at the waiter, other elegant guests laugh with him.",
   "narration_ar": "وكل ليلة، يجد من يسخر منه."
  },
  {
   "id": 4,
   "prompt": "Close-up portrait of {youssef} lowering his eyes, jaw tightening, lantern light on his face, emotional, shallow depth of field.",
   "narration_ar": "لكنه لا يردّ أبدًا... فهو يخفي سرًّا لا يعرفه أحد."
  },
  {
   "id": 5,
   "prompt": "{lalla_zahra} watches secretly from a carved cedar archway of {setting}, holding an old envelope against her chest, worried expression.",
   "narration_ar": "إلا امرأةً واحدة."
  },
  {
   "id": 6,
   "prompt": "Extreme close-up of an old envelope with a red wax seal bearing an ornate letter, resting on a carved wooden table, candle flame flickering, dramatic slow push-in.",
   "narration_ar": "والليلة... ستُفتح وصيةُ صاحب الرياض."
  }
 ]
}

In [ ]:
import os, time, json, gc, torch
from diffusers import WanPipeline, AutoencoderKLWan
from diffusers.utils import export_to_video

OUT = '/kaggle/working/dabreels_test'
os.makedirs(OUT, exist_ok=True)
cap = torch.cuda.get_device_capability(0)
DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16   # T4/P100 have no fast bf16

# Try the better model first, fall back to the small one if it does not fit.
CANDIDATES = [
    dict(name='Wan2.2-TI2V-5B', repo='Wan-AI/Wan2.2-TI2V-5B-Diffusers', w=704, h=1280, frames=73, fps=24, steps=30, guidance=5.0),
    dict(name='Wan2.1-T2V-1.3B', repo='Wan-AI/Wan2.1-T2V-1.3B-Diffusers', w=480, h=832, frames=65, fps=16, steps=30, guidance=5.0),
]

def load(c):
    vae = AutoencoderKLWan.from_pretrained(c['repo'], subfolder='vae', torch_dtype=torch.float32)
    pipe = WanPipeline.from_pretrained(c['repo'], vae=vae, torch_dtype=DTYPE)
    pipe.enable_model_cpu_offload()      # keeps only the active part on the GPU
    try:
        pipe.vae.enable_tiling()
    except Exception:
        pass
    return pipe

def fill(text):
    return text.format(setting=SCENE['setting'], **SCENE['characters'])

def render(pipe, c, shot):
    prompt = fill(shot['prompt']) + '. ' + SCENE['style']
    t = time.time()
    frames = pipe(prompt=prompt, negative_prompt=SCENE['negative'], height=c['h'], width=c['w'],
                  num_frames=c['frames'], guidance_scale=c['guidance'],
                  num_inference_steps=c['steps'],
                  generator=torch.Generator('cpu').manual_seed(1000 + shot['id'])).frames[0]
    import numpy as np
    arr = np.asarray(frames, dtype='float32')
    if not np.isfinite(arr).all() or arr.std() < 0.01:   # fp16 can give black frames
        raise ValueError('blank or broken output')
    path = f"{OUT}/shot_{shot['id']:02d}.mp4"
    export_to_video(frames, path, fps=c['fps'])
    return path, time.time() - t

report = {'gpu': torch.cuda.get_device_name(0), 'dtype': str(DTYPE), 'shots': []}
pipe, model = None, None
for c in CANDIDATES:
    try:
        print('Loading', c['name']); pipe = load(c)
        path, secs = render(pipe, c, SCENE['shots'][0])   # first shot doubles as the speed test
        model = c; report['model'] = c
        report['shots'].append({'id': 1, 'file': path, 'seconds': round(secs)})
        print(f"{c['name']}: shot 1 took {secs/60:.1f} min")
        break
    except torch.cuda.OutOfMemoryError as e:
        print(c['name'], 'does not fit on this GPU:', str(e)[:200])
    except Exception as e:
        print(c['name'], 'failed:', repr(e)[:300])
    pipe = None; gc.collect(); torch.cuda.empty_cache()
assert model, 'No model could run on this GPU'

In [ ]:
for shot in SCENE['shots'][1:]:
    path, secs = render(pipe, model, shot)
    report['shots'].append({'id': shot['id'], 'file': path, 'seconds': round(secs)})
    print(f"shot {shot['id']}: {secs/60:.1f} min")
json.dump(report, open(f'{OUT}/report.json', 'w'), indent=1)
print(json.dumps(report, indent=1))

In [ ]:
!cd /kaggle/working && zip -qr dabreels_test.zip dabreels_test && ls -la dabreels_test.zip